# Notebook 1 — Data Loading and Preprocessing

This notebook walks through the raw SSP-SP data, inspects its structure, and applies the preprocessing pipeline provided by the `vehicle_theft_network` package.

> **Note:** Run `uv sync` from the project root before opening this notebook so that the package is installed in the virtual environment.

## 1. Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd

from vehicle_theft_network.config import DataConfig
from vehicle_theft_network.data.loader import load_raw_data
from vehicle_theft_network.data.preprocessor import preprocess

cfg = DataConfig()

## 2. Load Raw Data

The raw data comes from the SSP-SP transparency portal ([link](https://www.ssp.sp.gov.br/transparenciassp)).
Three annual sheets (`VEICULOS_2022`, `VEICULOS_2023`, `VEICULOS_2024`) are concatenated.

In [1]:
raw = load_raw_data(cfg.raw_data_path, cfg.sheets)
raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 549995 entries, 0 to 549994
Data columns (total 52 columns):
 #   Column                    Non-Null Count   Dtype
---  ------                    --------------   -----
 0   ID_DELEGACIA              549995 non-null  int64
 1   NOME_DEPARTAMENTO         549995 non-null  object
 2   NOME_SECCIONAL            549995 non-null  object
 3   NOME_DELEGACIA            549995 non-null  object
 4   NOME_MUNICIPIO            549995 non-null  object
 5   ANO_BO                    549995 non-null  int64
 6   NUM_BO                    549995 non-null  object
 7   VERSAO                    333807 non-null  float64
 ...
 38  LATITUDE                  525348 non-null  float64
 39  LONGITUDE                 525348 non-null  float64
 ...
dtypes: datetime64[ns](3), float64(9), int64(3), object(37)
memory usage: 218.2+ MB


In [2]:
raw.columns

Index(['ID_DELEGACIA', 'NOME_DEPARTAMENTO', 'NOME_SECCIONAL', 'NOME_DELEGACIA',
       'NOME_MUNICIPIO', 'ANO_BO', 'NUM_BO', 'VERSAO',
       'NOME_DEPARTAMENTO_CIRC', 'NOME_SECCIONAL_CIRC', 'NOME_DELEGACIA_CIRC',
       'NOME_MUNICIPIO_CIRC', 'DATA_OCORRENCIA_BO', 'HORA_OCORRENCIA',
       'DESCRICAO_APRESENTACAO', 'DATAHORA_REGISTRO_BO', 'DATA_COMUNICACAO_BO',
       'DATAHORA_IMPRESSAO_BO', 'DESCR_PERIODO', 'AUTORIA_BO',
       'FLAG_INTOLERANCIA', 'TIPO_INTOLERANCIA', 'FLAG_FLAGRANTE',
       'FLAG_STATUS', 'DESC_LEI', 'FLAG_ATO_INFRACIONAL', 'RUBRICA',
       'DESCR_CONDUTA', 'DESDOBRAMENTO', 'CIRCUNSTANCIA', 'DESCR_TIPOLOCAL',
       'DESCR_SUBTIPOLOCAL', 'CIDADE', 'BAIRRO', 'CEP', 'LOGRADOURO_VERSAO',
       'LOGRADOURO', 'NUMERO_LOGRADOURO', 'LATITUDE', 'LONGITUDE',
       'CONT_VEICULO', 'DESCR_OCORRENCIA_VEICULO', 'DESCR_TIPO_VEICULO',
       'DESCR_MARCA_VEICULO', 'ANO_FABRICACAO', 'ANO_MODELO', 'PLACA_VEICULO',
       'DESC_COR_VEICULO', 'MES', 'ANO', 'CIDADE.1', 'DESC_NATU

## 3. Preprocessing Pipeline

The `preprocess` function chains four steps:
1. **Deduplication** — drop duplicate police reports (same `ANO_BO`, `NUM_BO`, `NOME_DELEGACIA`)
2. **Column selection** — keep only the seven relevant fields
3. **String normalisation** — strip diacritics, upper-case `BAIRRO` and `CIDADE`
4. **Coordinate / date filtering** — remove zero-coordinates and records outside 2022-01-01 → 2023-12-31

The parameters below match those used in the thesis. Adjust `cfg.start_date` / `cfg.end_date` or supply custom `columns_to_keep` to adapt the pipeline to a different study.

In [3]:
dados = preprocess(
    raw,
    columns=cfg.columns_to_keep,
    start_date=cfg.start_date,
    end_date=cfg.end_date,
)
print(f'{len(raw):,} raw records  ->  {len(dados):,} after preprocessing')
dados.info()

549,995 raw records  ->  276,503 after preprocessing
<class 'pandas.core.frame.DataFrame'>
Int64Index: 276503 entries, 0 to 304004
Data columns (total 9 columns):
 #   Column               Non-Null Count   Dtype
---  ------               --------------   -----
 0   BAIRRO               276502 non-null  object
 1   CIDADE               276503 non-null  object
 2   LATITUDE             276503 non-null  float64
 3   LONGITUDE            276503 non-null  float64
 4   DATA_OCORRENCIA_BO   276503 non-null  object
 5   HORA_OCORRENCIA      250669 non-null  object
 6   DESCR_MARCA_VEICULO  276503 non-null  object
 7   DATA_HORA            250669 non-null  datetime64[ns]
dtypes: datetime64[ns](1), float64(2), object(6)
memory usage: 19.0+ MB


## 4. Exploratory Analysis

### 4.1 City distribution

In [4]:
city_counts = dados['CIDADE'].value_counts()
city_counts.head(10)

S.PAULO                135559
CAMPINAS                14246
S.ANDRE                 14074
GUARULHOS               11597
S.BERNARDO DO CAMPO      9632
OSASCO                   8946
MAUA                     5615
DIADEMA                  5485
RIBEIRAO PRETO           5347
SOROCABA                 4668
Name: CIDADE, dtype: int64


In [5]:
# Cities that collectively cover 50% of all records
cumsum = city_counts.cumsum()
threshold = (cumsum / cumsum.max() >= 0.50).idxmax()
cities_50pct = city_counts.index[:city_counts.index.get_loc(threshold) + 1]
print('Cities covering 50% of records:')
print(cities_50pct)

Cities that describe 50.0 % of the instances:
Index(['S.PAULO', 'CAMPINAS', 'S.ANDRE', 'GUARULHOS'], dtype='object')


### 4.2 Unique neighbourhoods before and after normalisation

In [6]:
# String normalisation merges variants (e.g. 'São Paulo' / 'SAO PAULO' / 'Sao Paulo')
print('Unique BAIRRO values:', dados['BAIRRO'].nunique())
print('Unique CIDADE values:', dados['CIDADE'].nunique())

Unique BAIRRO values: 16447
Unique CIDADE values: 627


## 5. Save Processed Data

In [ ]:
dados.to_parquet(cfg.processed_parquet, index=False)
print(f'Saved {len(dados):,} records to {cfg.processed_parquet}')

---

## Appendix: Neighbourhood Enrichment via Shapefiles (optional)

During the original study, district polygons from municipal open-data portals were used to correct `BAIRRO` labels for São Paulo, São Bernardo do Campo, and Guarulhos. This step is **not** part of the main pipeline — the pipeline uses the raw `BAIRRO` strings for metadata only; spatial analysis is performed exclusively through the grid.

The code below is kept for reference. To run it you need the shapefiles downloaded from the respective city portals.

In [ ]:
# import geopandas as gpd
# from shapely.geometry import Point

# shapefile_path = 'path/to/SIRGAS_SHP_distrito.shp'
# gdf = gpd.read_file(shapefile_path).set_crs('EPSG:29193').to_crs(epsg=4326)

# for index, row in dados.iterrows():
#     if row['CIDADE'] == 'S.PAULO':
#         pt = Point(row['LONGITUDE'], row['LATITUDE'])
#         match = next((r['ds_nome'] for _, r in gdf.iterrows()
#                       if r['geometry'].contains(pt)), None)
#         if match:
#             dados.at[index, 'BAIRRO'] = match